# Generate loan_details.csv

This notebook reads `data/customer_profile.csv` and generates `data/loan_details.csv` with a configurable number of loans (default 12,000).
It samples customers with weighted probabilities so gig workers and self-employed customers receive proportionally more loans, then assigns products and loan terms following Stage 3 rules.

In [26]:
# Imports and parameters
import numpy as np
import pandas as pd
from pathlib import Path
import random

# Parameters - adjust as needed
TOTAL_LOANS = 12000
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

cust_path = Path('../data/customer_profile.csv')
if not cust_path.exists():
    raise FileNotFoundError('Run 01_generate_customers.ipynb first to create data/customer_profile.csv')
cp = pd.read_csv(cust_path)
N = len(cp)

print(f'Configured TOTAL_LOANS={TOTAL_LOANS}, SEED={SEED}')
print(f'Loaded {N} customers from {cust_path}')

Configured TOTAL_LOANS=12000, SEED=42
Loaded 10000 customers from ..\data\customer_profile.csv


In [27]:
# Build sampling weights: base weight 1, boost for gig & self-employed
weights = np.ones(N, dtype=float)
weights += cp['employment_type'].apply(lambda x: 0.9 if x in ['Gig Worker','Self Employed'] else 0.0).values
# normalize to probabilities
probs = weights / weights.sum()
# Sample customer indices for TOTAL_LOANS (allows repeats = multiple loans per customer)
chosen_indices = np.random.choice(np.arange(N), size=TOTAL_LOANS, replace=True, p=probs)
# create ordered list of customer_ids for loans
customer_ids_for_loans = cp['customer_id'].values[chosen_indices].tolist()
print(f'Sampled {len(customer_ids_for_loans)} loans mapped to customers')

# Quick check: distribution of loans per customer (top few)
import collections
cnts = collections.Counter(customer_ids_for_loans)
print('Top customers by loan count (sample):', cnts.most_common(5))

Sampled 12000 loans mapped to customers
Top customers by loan count (sample): [('CUST005577', 7), ('CUST005039', 7), ('CUST000399', 7), ('CUST008746', 7), ('CUST004120', 7)]


In [28]:
# Build sampling weights: base weight 1, boost for gig & self-employed
weights = np.ones(N, dtype=float)
weights += cp['employment_type'].apply(lambda x: 0.9 if x in ['Gig Worker','Self Employed'] else 0.0).values
# normalize to probabilities
probs = weights / weights.sum()
# Sample customer indices for TOTAL_LOANS (allows repeats = multiple loans per customer)
chosen_indices = np.random.choice(np.arange(N), size=TOTAL_LOANS, replace=True, p=probs)
# create ordered list of customer_ids for loans
customer_ids_for_loans = cp['customer_id'].values[chosen_indices].tolist()
print(f'Sampled {len(customer_ids_for_loans)} loans mapped to customers')

# Quick check: distribution of loans per customer (top few)
import collections
cnts = collections.Counter(customer_ids_for_loans)
print('Top customers by loan count (sample):', cnts.most_common(5))

Sampled 12000 loans mapped to customers
Top customers by loan count (sample): [('CUST008990', 8), ('CUST007794', 8), ('CUST003889', 8), ('CUST006195', 8), ('CUST002929', 7)]


In [29]:
# Helper functions for product assignment and loan terms
def choose_product(emp):
    if emp == 'Student':
        return np.random.choice(['Education Loan','BNPL','Personal Loan'], p=[0.85,0.10,0.05])
    if emp == 'Gig Worker':
        return np.random.choice(['Personal Loan','BNPL','SME Loan'], p=[0.6,0.35,0.05])
    if emp == 'Business Owner':
        return np.random.choice(['SME Loan','Personal Loan'], p=[0.8,0.2])
    if emp == 'Self Employed':
        return np.random.choice(['Personal Loan','SME Loan','BNPL'], p=[0.6,0.25,0.15])
    return np.random.choice(['Personal Loan','BNPL','SME Loan'], p=[0.7,0.2,0.1])  # Salaried

def loan_terms(product):
    if product == 'Personal Loan':
        amount = int(np.random.uniform(10000,500000))
        tenure = int(np.random.randint(6,37))
        interest = round(np.random.uniform(16,32),2)
    elif product == 'Education Loan':
        amount = int(np.random.uniform(100000,1000000))
        tenure = int(np.random.randint(12,49))
        interest = round(np.random.uniform(5,10),2)
    elif product == 'BNPL':
        amount = int(np.random.uniform(2000,60000))
        tenure = int(np.random.randint(2,11))
        interest = round(np.random.uniform(0,14),2)
    else:  # SME Loan
        amount = int(np.random.uniform(50000,600000))
        tenure = int(np.random.randint(8,33))
        interest = round(np.random.uniform(16,30),2)
    return amount, tenure, interest

def risk_grade(score):
    try:
        s = float(score)
    except Exception:
        return 'Unknown'
    if s >= 750:
        return 'A'
    if s >= 650:
        return 'B'
    if s >= 550:
        return 'C'
    return 'D'

In [30]:
# Build loan records; use a mapping for fast customer lookups
cp_index = {row['customer_id']: row for _, row in cp.iterrows()}
loans = []
for i, cid in enumerate(customer_ids_for_loans):
    cust = cp_index[cid]
    emp = cust['employment_type']
    product = choose_product(emp)
    amount, tenure, interest = loan_terms(product)
    rate_monthly = interest/100/12
    if rate_monthly > 0:
        emi = amount * rate_monthly / (1 - (1+rate_monthly)**(-tenure))
    else:
        emi = amount/tenure
    emi = round(float(emi),2)
    score = cust.get('credit_score', None)
    rgrade = risk_grade(score)
    onboard = pd.to_datetime(cust.get('onboarding_date')) if 'onboarding_date' in cust else pd.to_datetime('2022-01-01')
    start = onboard
    end = pd.to_datetime('2023-12-31')
    # pick origination between onboarding and end; if onboarding after end, use onboarding
    try:
        if start > end:
            orig_date = start
        else:
            orig_date = pd.to_datetime(np.random.choice(pd.date_range(start, end)))
    except Exception:
        orig_date = pd.to_datetime('2023-01-01')
    disb_date = orig_date + pd.Timedelta(days=int(np.random.randint(1,15)))
    processing_fee = round(amount * np.random.uniform(0.005, 0.02),2)
    co_borrower = 1 if (product == 'Education Loan' and amount > 300000 and random.random() < 0.3) else 0
    loans.append({
        'loan_id': f'LOAN{str(i+1).zfill(7)}',
        'customer_id': cid,
        'product_type': product,
        'loan_amount': amount,
        'tenure_months': int(tenure),
        'interest_rate': interest,
        'emi_amount': emi,
        'risk_grade': rgrade,
        'origination_date': orig_date.date().isoformat(),
        'processing_fee': processing_fee,
        'loan_purpose': product,
        'disbursement_date': disb_date.date().isoformat(),
        'co_borrower_flag': co_borrower
    })

loan_df = pd.DataFrame(loans)
print('Generated loans dataframe with rows:', len(loan_df))

# Save to data folder
Path('../data').mkdir(parents=True, exist_ok=True)
out_path = Path('../data/loan_details.csv')
loan_df.to_csv(out_path, index=False)
print(f'Wrote {len(loan_df)} loans to: {out_path}')

Generated loans dataframe with rows: 12000
Wrote 12000 loans to: ..\data\loan_details.csv


In [31]:
# Quick validations
print('Product distribution:')
print(loan_df['product_type'].value_counts(normalize=True).round(3))
print('Risk grade distribution:')
print(loan_df['risk_grade'].value_counts(normalize=True).round(3))
print('Loans per customer (top 10):')
print(loan_df.groupby('customer_id').size().sort_values(ascending=False).head(10))
print('Sample rows:')
print(loan_df.head().to_string(index=False))

Product distribution:
product_type
Personal Loan     0.560
BNPL              0.219
SME Loan          0.162
Education Loan    0.060
Name: proportion, dtype: float64
Risk grade distribution:
risk_grade
C    0.346
B    0.319
A    0.182
D    0.153
Name: proportion, dtype: float64
Loans per customer (top 10):
customer_id
CUST007794    8
CUST008990    8
CUST006195    8
CUST003889    8
CUST009851    7
CUST007756    7
CUST009223    7
CUST002929    7
CUST003009    7
CUST009822    6
dtype: int64
Sample rows:
    loan_id customer_id  product_type  loan_amount  tenure_months  interest_rate  emi_amount risk_grade origination_date  processing_fee  loan_purpose disbursement_date  co_borrower_flag
LOAN0000001  CUST006553          BNPL        29119              3          13.63     9927.66          C       2023-07-02          541.71          BNPL        2023-07-10                 0
LOAN0000002  CUST000807 Personal Loan       173231             26          21.15     8363.06          A       2023-10-31  